In [ ]:
# ── Title ─────────────────────────────────────────────
# MedAssist AI — Notebook 1: Data Exploration & Indexing 
# ─────────────────────────────────────────────────────────────
# What this notebook does:
#   1. Loads PubMedQA pqa_artificial (~211k records) as the retrieval corpus
#   2. Loads pqa_labeled (1k records) as the future evaluation set
#   3. Uses a strong biomedical embedding model (S-PubMedBert-MS-MARCO)
#   4. Chunks documents efficiently with batching
#   5. Builds and saves FAISS index 

In [ ]:
# ── Cell 2: Install ───────────────────────────────────────────
!pip install -q --force-reinstall "numpy==1.26.4" "pandas==2.2.2"
!pip install -q "langchain==0.2.16" "langchain-community==0.2.16"
!pip install -q "langchain-huggingface==0.0.3" "faiss-cpu==1.8.0"
!pip install -q "sentence-transformers==3.0.1" "datasets==2.21.0"
!pip install -q "tqdm==4.66.5"

import IPython
IPython.Application.instance().kernel.do_shutdown(restart=True)

In [ ]:
# ── Cell 3: Imports + Config ──────────────────────────────────
import numpy as np
import pandas as pd
from tqdm import tqdm
from datasets import load_dataset
from langchain.docstore.document import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

CONFIG = {
    "embedding_model": "pritamdeka/S-PubMedBert-MS-MARCO",   
    "chunk_size": 512,
    "chunk_overlap": 64,
    "batch_size": 256,                                        
}

print(" Imports ready")

In [ ]:
# ── Cell 4: Load Datasets ──────────────
print("Loading PubMedQA datasets...")

# Retrieval corpus — large artificial set (~211k)
dataset_artificial = load_dataset("pubmed_qa", "pqa_artificial", split="train")
print(f" Loaded pqa_artificial: {len(dataset_artificial):,} records (retrieval corpus)")

# Evaluation set — labeled only (will NOT be indexed)
dataset_labeled = load_dataset("pubmed_qa", "pqa_labeled", split="train")
print(f" Loaded pqa_labeled:   {len(dataset_labeled):,} records (evaluation only)")

# Quick stats
print("\n Decision distribution in evaluation set (pqa_labeled):")
df_labeled = pd.DataFrame(dataset_labeled)
print(df_labeled["final_decision"].value_counts())

In [ ]:
# ── Cell 5: Preprocess & Chunk Documents ──────────────────────
def records_to_documents(dataset, is_artificial=True):
    documents = []
    for record in tqdm(dataset, desc="Converting to Documents", total=len(dataset)):
        ctx = record.get("context", {})
        if isinstance(ctx, dict) and "contexts" in ctx:
            text = " ".join(ctx["contexts"])
        else:
            text = str(ctx)

        if len(text.strip()) < 50:
            continue

        documents.append(Document(
            page_content=text,
            metadata={
                "doc_id": len(documents),
                "question": record.get("question", ""),
                "gold_answer": record.get("long_answer", "")[:300],
                "decision": record.get("final_decision", "") if not is_artificial else "artificial",
                "source": f"PubMedQA_artificial_{len(documents)}" if is_artificial else f"PubMedQA_labeled_{record.get('pub_id', len(documents))}",
            }
        ))
    return documents


print("Creating documents from pqa_artificial (retrieval corpus only)...")
raw_docs = records_to_documents(dataset_artificial, is_artificial=True)
print(f" {len(raw_docs):,} documents created from artificial corpus")

# Chunking
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CONFIG["chunk_size"],
    chunk_overlap=CONFIG["chunk_overlap"],
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = splitter.split_documents(raw_docs)
print(f" {len(chunks):,} chunks created")
print(f" Average chunk length: {np.mean([len(d.page_content) for d in chunks]):.0f} characters")

In [ ]:
# ── Cell 6: Optimized Embedding + FAISS Indexing ─────────────────────────────
import gc
import os
import torch
import numpy as np
import faiss
import psutil
import shutil
from tqdm import tqdm
from langchain.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS as LangchainFAISS

# ── Clear any previous run from memory ───────────────────────────────────────
for var in ["embedding_model", "embeddings", "raw_embeddings", "index", "vectorstore", "docstore", "texts"]:
    if var in globals():
        del globals()[var]
gc.collect()
torch.cuda.empty_cache()
print(" Memory cleared")

# ── Device Setup ──────────────────────────────────────────────────────────────
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f" Using device: {device.upper()}")

ram = psutil.virtual_memory()
print(f" RAM at start — Used: {ram.used / 1024**3:.1f} GB / Total: {ram.total / 1024**3:.1f} GB")
# ── Embedding Model ───────────────────────────────────────────────────────────
EMBED_BATCH_SIZE = CONFIG["batch_size"]

embedding_model = HuggingFaceEmbeddings(
    model_name=CONFIG["embedding_model"],
    model_kwargs={"device": device},
    encode_kwargs={
        "normalize_embeddings": True,
        "batch_size": EMBED_BATCH_SIZE,
    },
)

dimension = len(embedding_model.embed_query("test"))
print(f" Embedding dimension: {dimension}")

# ── Compute Embeddings  ──────────────────────────────────────
texts = [doc.page_content for doc in chunks]
total_batches = (len(texts) + EMBED_BATCH_SIZE - 1) // EMBED_BATCH_SIZE  # ceiling division

print(f" Computing embeddings for {len(texts):,} chunks in {total_batches} batches...")
raw_embeddings = []

with tqdm(total=len(texts), desc="Embedding", unit="doc") as pbar:
    for i in range(0, len(texts), EMBED_BATCH_SIZE):
        batch = texts[i : i + EMBED_BATCH_SIZE]
        batch_embeddings = embedding_model.embed_documents(batch)
        raw_embeddings.extend(batch_embeddings)
        pbar.update(len(batch))
        pbar.set_postfix({
            "batch": f"{i // EMBED_BATCH_SIZE + 1}/{total_batches}",
            "docs_done": f"{min(i + EMBED_BATCH_SIZE, len(texts)):,}/{len(texts):,}"
        })

embeddings = np.array(raw_embeddings, dtype=np.float32)
print(f" All embeddings computed — shape: {embeddings.shape}")



In [ ]:
# ── Build FAISS Index ─────────────────────────────────────────────────────────
print(" Building FAISS index...")
index = faiss.IndexFlatIP(dimension)  
index.add(embeddings)
print(f"FAISS index built — {index.ntotal:,} vectors")

In [ ]:
# ── Docstore + ID Mapping ─────────────────────────────────────────────────────
docstore = InMemoryDocstore({str(i): doc for i, doc in enumerate(chunks)})
index_to_docstore_id = {i: str(i) for i in range(len(chunks))}

# ── Wrap into LangChain FAISS Vector Store ────────────────────────────────────
vectorstore = LangchainFAISS(
    embedding_function=embedding_model,
    index=index,
    docstore=docstore,
    index_to_docstore_id=index_to_docstore_id,
)
print(" VectorStore ready")


In [ ]:
# ── Persist to Disk ───────────────────────────────────────────────────────────
vectorstore.save_local("/kaggle/working/faiss_index")
print("✅ FAISS index saved to: /kaggle/working/faiss_index")